In [1]:
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from openai import OpenAI

In [2]:

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()
# MODEL = 'llama3.2'
# OLLAMA_BASE_URL = "http://localhost:11434/v1"
# ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key="ollama")

API key looks good so far


In [3]:
from bs4 import BeautifulSoup
import requests


# Standard headers to fetch a website
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/117.0.0.0 Safari/537.36"
}


def fetch_website_contents(url):
    """
    Return the title and contents of the website at the given url;
    truncate to 2,000 characters as a sensible limit
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    title = soup.title.string if soup.title else "No title found"
    if soup.body:
        for irrelevant in soup.body(["script", "style", "img", "input"]):
            irrelevant.decompose()
        text = soup.body.get_text(separator="\n", strip=True)
    else:
        text = ""
    return (title + "\n\n" + text)[:2_000]


def fetch_website_links(url):
    """
    Return the links on the webiste at the given url
    I realize this is inefficient as we're parsing twice! This is to keep the code in the lab simple.
    Feel free to use a class and optimize it!
    """
    response = requests.get(url, headers=headers)
    soup = BeautifulSoup(response.content, "html.parser")
    links = [link.get("href") for link in soup.find_all("a")]
    return [link for link in links if link]

In [4]:
links = fetch_website_links("https://edwarddonner.com")
links

['https://edwarddonner.com/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/11/11/ai-live-event/',
 'https://edwarddonner.com/2025/09/15/ai-in-production-gen-ai-and-agentic-ai-on-aws-at-scale/',
 'https://edwarddonner.com/2025/09/1

In [5]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [6]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

https://edwarddonner.com/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/
https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/

In [8]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    # response = ollama.chat.completions.create(
    #     model=MODEL,
    #     messages=[
    #         {"role": "system", "content": link_system_prompt},
    #         {"role": "user", "content": get_links_user_prompt(url)}
    #     ],
    #     response_format={"type": "json_object"}
    # )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [9]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'}]}

In [10]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [11]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [12]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 7 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'company page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'social media - twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'code repository', 'url': 'https://github.com/huggingface'},
  {'type': 'community forum', 'url': 'https://discuss.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

# Preparing  the brochure

In [13]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [14]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links
## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-OCR
Updated
5 days ago
•
248k
•
804
Qwen/Qwen3-Coder-Next
Updated
5 days ago
•
76.6k
•
598
moonshotai/Kimi-K2.5
Updated
3 days ago
•
405k
•
1.85k
stepfun-ai/Step-3.5-Flash
Updated
1 day ago
•
105k
•
520
openbmb/MiniCPM-o-4_5
Updated
about 21 hours ago
•
17.5k
•
605
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.31k
Qwen3-TTS Demo
🎙
1.31k
Transform text into natural-sounding speech with custom voices
Running
on
A100
190
ACE-Step v1.5
🎵
190
Music Generation Foundation Model v1.5
Running
486
Demo Playground
⚡
486
Free pl

In [ ]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [16]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [17]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 4 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nCommunity\nDocs\nEnterprise\nPricing\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nzai-org/GLM-OCR\nUpdated\n5 days ago\n•\n248k\n•\n804\nQwen/Qwen3-Coder-Next\nUpdated\n5 days ago\n•\n76.6k\n•\n598\nmoonshotai/Kimi-K2.5\nUpdated\n3 days ago\n•\n405k\n•\n1.85k\nstepfun-ai/Step-3.5-Flash\nUpdated\n1 day ago\n•\n105k\n•\n520\nopenbmb/MiniCPM-o-4_5\nUpdated\nabout 21 hours ago\n•\n17.5k\n•\n605\nBrowse 2M+ models\nSpaces\nRunning\non\nZero\nFeatured\n1.31k\nQwen3-TTS Demo\n🎙

In [18]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [19]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 8 relevant links


# Welcome to Hugging Face: The AI Community That’s Hugging the Future!

---

## Who Are We?

Hugging Face is not just a company — it’s a bustling hub of AI enthusiasts, machine learning engineers, data scientists, and curious minds who believe that open and ethical AI is the future. We are the “home of machine learning” where collaboration meets innovation, and where 2 million+ models, hundreds of thousands of datasets, and thousands of AI apps come alive.

---

## What Do We Do?

- **Host & Collaborate:** Share, explore, and collaborate on unlimited public ML models, datasets, and cool applications.
- **Accelerate ML:** Power your projects with our open-source stack or supercharge your enterprise needs with paid Compute and secure Enterprise solutions.
- **Explore All Modalities:** Text, images, video, audio, *or even 3D*—we support it all.
- **Build Your Machine Learning Portfolio:** Be recognized by the world for your AI creations. Share, showcase, and celebrate your work.

---

## The Heartbeat of Our Community

We’re open-source fanatics with a **fast-growing global community**, weaving the story of AI with transparency, ethics, and a touch of fun. You get to hang out in our vibrant forums, contribute to AI models, take part in Spaces (live demos and collaborative playgrounds), or just geek out over the latest AI breakthroughs.

Whether you’re a newbie wanting to explore sweet voice-to-speech magic or a seasoned AI wizard deploying enterprise-grade applications, Hugging Face is your playground.

---

## Cool Stuff Trending This Week

- **zai-org/GLM-OCR:** Optical Character Recognition model updated just 5 days ago with 248k downloads! Perfect for reading between the lines.
- **Qwen/Qwen3-Coder-Next:** AI helping coders level up with almost 77k downloads!
- **moonshotai/Kimi-K2.5:** AI with a whopping 405k downloads—because who doesn’t love a moonshot?
- And many more – browsing 2 million+ models keeps your AI thrills going.

---

## Culture & Vibes

If you like a workplace buzzing with **curiosity, collaboration, and a splash of geekery**, Hugging Face might just be your next home. We champion:

- **Open source passion** — because sharing is caring.
- **Innovation at breakneck speed.**
- **Ethical AI** — building the future responsibly.
- Working alongside some of the **brightest minds** at the edge of technology.

---

## Careers: Want to Join the Fun?

Looking for AI engineering gigs, open-source wizardry, or enterprise solutions smarts? Hugging Face is hiring! We’re always on the lookout for:

- Machine Learning Engineers
- Data Scientists
- Software Developers
- Community Builders & Evangelists

If you dream in Python and sip on creativity, check out our full list of openings and join us in shaping the future.

---

## Why Hugging Face? Because We Are More Than Models

- **Community-powered:** You’re not alone in the AI race, you’re part of a vibrant team of collaborators worldwide.
- **Open-source heaven:** Plug and play with the tools and tech loved by the AI community.
- **Enterprise-ready:** Need rock-solid security and control? We've got your back.
- **Innovation galore:** From text to audio, video, and 3D — stick with us, and you’re in for a wild ride.

---

## Laugh a Little

Why does the AI model go to therapy? Because it had too many layers to unpack! 

At Hugging Face, we get it — sometimes AI is serious, but we know how to keep it light and friendly.

---

## Ready to Hug the Future?

Join us at **[huggingface.co](https://huggingface.co/)** and start building the AI you always wanted.

---

*Powered by smiles, code, and the collective dream to make AI accessible and fun.*  
**Hugging Face™ — Where AI meets Community!**  
#AIwithHeart #OpenSourceLove #JoinTheHug

---

**Follow us:** GitHub | Twitter | LinkedIn | Discord  
**Contact:** Careers | Docs | Community Forums | Enterprise Solutions

# with Stream

In [20]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [21]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# Welcome to Hugging Face: The AI Playground for Brainy Builders and Curious Coders!

---

## Who Are We?

At Hugging Face, we're not just a company — we're the **bustling community where machine learning dreams come to life**. Think of us as the AI version of a neighborhood potluck party, but instead of casseroles, everyone brings *models, datasets,* and *mind-blowing applications* to share and collaborate on.

- Over **2 million models** and **500,000 datasets** available. (Yes, we took going big seriously.)
- A spicy mix of AI magic: text, image, video, audio, and even sneaky 3D models.
- A platform buzzing with bustling spaces to demo projects you’ll brag about on social media.

---

## What’s Cooking?

### Models 🎩✨  
From OCR that reads your messy handwriting to text-to-speech engines that’ll make Siri jealous, our trending models have thousands of fans and growing:

- **zai-org/GLM-OCR** — conquering digits and letters alike
- **Qwen3-TTS Demo** — transform your texts into *smooth* and natural voices
- **Z Image Turbo** — conjure stunning images from mere text, like magic!

### Datasets & Spaces: The AI Irony Buffet 🍲  
Dump your data, grab a dataset, or host a space to showcase your genius project. Whether you're training a model or just testing your AI wit, there’s a spot here with your name on it.

---

## Who’s Our Crowd?

We’re a *global village* — from solo researchers to enterprise juggernauts, from university labs to startups burning the midnight oil to launch new apps. If it involves AI, **it’s happening here**.

- Community members share **build portfolios** to flaunt their AI street cred.
- Enterprises rely on our **secure, scalable, and enterprise-grade solutions** to juggle serious AI workloads safely.
- Startups and hobbyists enjoy **free and paid compute** options that turbocharge their projects.

---

## Why Work Here? (Spoiler: Because It's Fun)

If you want a job where your *sixth sense for AI* gets to party with brilliant minds — Hugging Face is your playground!

- Culture? We blend *open-source hustle* with *community spirit* — every voice counts, whether you're writing code or troubleshooting models.
- Career paths? From *engineers, data scientists,* to *AI whisperers* — get ready to learn, teach, and evolve.
- Perks? Build the future with a team as passionate about AI as you are, with flexible environments made for true innovation.

---

## Enterprise Love 💼❤️

If you’re a team buying the *future of AI* in bulk:

- Enterprise-grade security? Check.
- Single Sign-On and audit logs? Double check.
- Dedicated support and flexible contracts? Triple check.

Let us handle the heavy AI lifting while you focus on making the next big thing.

---

## Join the AI Revolution — It’s Hugging Time! 🤗

Whether you’re a curious coder, a data diva, or a business boss, **Hugging Face is your hub** to build smarter, faster, and cooler AI.

**Sign up today** and become part of the community that's not just building AI models — we’re *building the future* itself.

---

### Explore More  
- Browse 2 million+ models  
- Dig into 500k+ datasets  
- Try the coolest AI demos (did someone say music generation and video from text?)  

Ready? The future’s hugging you already.  

---

_Hugging Face · The AI community where brains meet bytes and brilliance happens._

In [22]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 11 relevant links


# 🤗 Hugging Face: The AI Community That's Hugging the Future 🤗

Welcome to **Hugging Face**, where the future of machine learning is built one friendly commit at a time! Picture an AI community that's less about cold code and more about warm collaboration. Whether you're a model magician, data diva, or just AI-curious — this is your playground, your lab, your canvas.

---

## Why Hugging Face? Because AI Shouldn't Be Lonely

At Hugging Face, we believe machine learning shouldn’t happen in a black box... it’s a vibrant, buzzing community of 80,000+ AI enthusiasts orbiting the globe. Here’s what makes us special:

- **2 Million+ Models:** From quirky text generators to image wizards and speech transformers, we’ve got AI models for anything and everything (well, almost).
- **500k+ Datasets:** Feast your algorithms on a treasure trove of data, continuously updated by passionate contributors.
- **Spaces – Your AI Playground:** Spin up app demos and share your AI art, music, or stories. If it involves AI, it lives here.
- **Community Spirit:** Collaboration beats competition. Share your work, build your ML portfolio, and know you’re part of *the* hub for open-source AI.
- **Multi-Modal Magic:** Text, images, sound, video, 3D — we don’t discriminate against data types, we embrace them all!

---

## The Hugging Face Culture: Be Open, Be Friendly, Hug Lots!

We’re not just a company; we’re a movement. Our culture is about:

- **Accessibility:** Democratizing good machine learning for everyone. No secret handshakes, just open source love.
- **Learning & Sharing:** Building expertise with kindness — from beginners to experts, there’s a place and pace for all.
- **Diversity & Inclusion:** Machines learn better from diverse inputs, and so do we. Different voices make us smarter.
- **Innovation With Ethics:** Pioneering responsible AI development that respects humans and their data — with a smile.

Join a team of 189 brilliant minds (and growing!) eager to change the world through collaboration, creativity, and a bit of AI wizardry.

---

## Careers — Your Future is Hugging Face Shaped

Feel like your current job is all work and no play? Come join us! We’re on a mission to democratize machine learning and are looking for:

- ML engineers who speak code *and* human
- Data scientists ready to dig deep and surface insights
- Community managers born to unite creators and coders
- Dreamers and doers eager to build the next AI breakthroughs

**Perks?** Work with cutting-edge tech, an insanely supportive community, enterprise-grade solutions, and the chance to truly make an impact.

---

## Customers? Oh, We Hug ‘Em All

- **Enterprises** craving secure, scalable AI solutions.
- **Researchers** pushing the boundaries of what AI can do.
- **Developers & Scientists** who want a seamless, collaborative platform.
- **Educators & Learners** diving into the new language of AI.

Hugging Face empowers professionals, hobbyists, and everyone in between to innovate faster, smarter, and friendlier.

---

## Get in Touch & Join the Party

- Explore AI models today at [huggingface.co](https://huggingface.co)
- Dive into datasets and apps for your next project
- Sign up for free or step up to paid Compute and Enterprise-grade platforms
- Follow the AI grapevine and join 80,000+ AI enthusiasts growing the AICA (AI Community of Awesome)

---

### Hugging Face — Where AI Feels Like Home 🤗  
Because the future of machine learning is better when shared. Hug. Collaborate. Build.

*P.S. Our signature yellow (#FFD21E) is the official color of AI sunshine and good vibes.*

---

**Are you ready for your next AI adventure?**  
*Come hug the future with us!*